In [25]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

import numpy as np
from sentence_transformers import SentenceTransformer

In [26]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
# Encode query
q = "Can I still join the course after the start date?"
v1 = model.encode(q)

qq = "How to isntall docker on windows?"
v2 = model.encode(qq)

# Encode document (one of the documents, just for an example)
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [28]:
# for cosin similarity we use dot only because the model normalizes the vectors. 
np.dot(v2,dv)

np.float32(0.019688543)

In [29]:
# we take Q and compare it against all Doc: get top 5 (candidates)
from ingest import load_faq_data

documents = load_faq_data() # dict 
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [30]:
# conv dict(documents) to text 
texts = []

for doc in documents: 
    text = doc['question'] + ' ' + doc["answer"]
    texts.append(text)

In [31]:
# We split into batches because text is long, we embed each chunk then embed into vector space 
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [32]:
# turn into 2d array (matrix) where rows = doc/vectors and col = dim of vectors
X = np.array(vectors) # our documents 
X.shape

(1350, 384)

In [38]:
query = "I just discovered the course, can I still join?"
v_query = model.encode(query)

In [39]:
scores = X.dot(v_query)
print(scores)

[ 0.41303527  0.2611987   0.60880876 ... -0.04739503  0.08148272
 -0.02353703]


In [40]:
# we select the closest want 
idx = np.argmax(scores) # idx of the largest 
idx, scores[idx]

(np.int64(538), np.float32(0.83177906))

In [41]:
documents[538]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [61]:
# get top 5 score 
top5 = np.argsort(-scores)[:5]
print(top5) # start, stop, step

[538 907 625   2 503]


In [ ]:
for idx in top5: 
    print(scores[idx])
    print(documents[idx])
    print()

0.83177906
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.68456966
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'doc_id': '41aabbd7c5'}

0.61755615
{'course': 'mlops-zoomcamp', 'section': '